**1. Import Libraries and Mount to Drive**

In [ ]:
# import libraries
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from huggingface_hub import login
from datasets import get_dataset_config_names, load_dataset, concatenate_datasets
from transformers import AutoTokenizer
from google.colab import drive

# mount to personal google drive
drive.mount('/content/drive')

Mounted at /content/drive


**2. Download & Export Dataset**

In [ ]:
# print message to show dataset processing has started for dataset one
print(f"\n--- Processing dataset: AlpaCare-MedInstruct-52k ---")

# load and process AlpaCare dataset
ds_alpac = load_dataset("lavita/AlpaCare-MedInstruct-52k", split="train")
ds_alpac = ds_alpac.add_column("dataset_source", ["AlpaCare-52k"] * len(ds_alpac))
print(f"    - Total Number of Rows: {len(ds_alpac)}")
print(f"    - Columns: {ds_alpac.column_names}")

# print message to show dataset processing has started for dataset two
print(f"\n--- Processing dataset: bioinstruct ---")

# load and process Bioinstruct dataset
ds_bio = load_dataset("bio-nlp-umass/bioinstruct", split="train")
ds_bio = ds_bio.add_column("dataset_source", ["Bioinstruct"] * len(ds_bio))
print(f"    - Total Number of Rows: {len(ds_bio)}")
print(f"    - Columns: {ds_bio.column_names}")

# merge both datasets into one
ds_final = concatenate_datasets([ds_bio, ds_alpac])

# shuffle the combined dataset
ds_final = ds_final.shuffle(seed=42)

# confirm dataset processing is done and show final size
print(f"\n--- Dataset Processing Completed ---")
print(f"    - Dataset final size: {len(ds_final)}")
print(f"    - Columns: {ds_final.column_names}")

# save final dataset to jsonl format
output_file = "medical_training_data.jsonl"
ds_final.to_json(output_file, lines=True, force_ascii=False)

# confirm the file has been saved and show its name
print(f"--- Completed! Saved to {output_file}")


--- Processing dataset: AlpaCare-MedInstruct-52k ---


README.md:   0%|          | 0.00/944 [00:00<?, ?B/s]

data/train-00000-of-00001-297892d5d4e8a0(…):   0%|          | 0.00/36.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/52002 [00:00<?, ? examples/s]

    - Total Number of Rows: 52002
    - Columns: ['output', 'input', 'instruction', 'dataset_source']

--- Processing dataset: bioinstruct ---


...

biomed_instruct_25k.json:   0%|          | 0.00/13.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25005 [00:00<?, ? examples/s]

    - Total Number of Rows: 25005
    - Columns: ['instruction', 'input', 'output', 'dataset_source']

--- Dataset Processing Completed ---
    - Dataset final size: 77007
    - Columns: ['instruction', 'input', 'output', 'dataset_source']


Creating json from Arrow format:   0%|          | 0/78 [00:00<?, ?ba/s]

--- Completed! Saved to medical_training_data.jsonl


**3. Remove Duplicates**

In [ ]:
# loads the json file into a variable called ds
ds = load_dataset("json", data_files="medical_training_data.jsonl", split="train")

# turns the dataset into a panda table
df = pd.DataFrame(ds)

# creates another copy of the dataframe
unprocessed_duplicates_df = df.copy()

# gets the length of the unprocessed dataframe
unprocessed_duplicates_ds = len(unprocessed_duplicates_df)

# finds duplicate rows and gets their index (row number) based on content columns only (ignoring source column)
duplicated_rows_unprocessed = unprocessed_duplicates_df.index[unprocessed_duplicates_df.duplicated(subset=["instruction", "input", "output"])].tolist()

# gets the length of the duplicate rows
duplicates_unprocessed = len(duplicated_rows_unprocessed)

# drops the duplicated rows based on content columns only (ignoring source column)
processed_duplicates_df = unprocessed_duplicates_df.drop_duplicates(subset=["instruction", "input", "output"])

# gets the length of the processed dataframe
processed_duplicates_ds = len(processed_duplicates_df)

# prints a summary of the deduplication process
print(f"--- Deduplication Summary ---")
print(f"    - Rows before : {unprocessed_duplicates_ds}")
print(f"    - Duplicates found : {duplicates_unprocessed}")
print(f"    - Rows after : {processed_duplicates_ds}")
print(f"    - Rows removed : {unprocessed_duplicates_ds - processed_duplicates_ds}")

Generating train split: 0 examples [00:00, ? examples/s]

--- Deduplication Summary ---
    - Rows before : 77007
    - Duplicates found : 0
    - Rows after : 77007
    - Rows removed : 0


**4. Remove Missing Rows (instruction & output)**

In [ ]:
# creates another copy of the dataframe
unprocessed_missing_df = processed_duplicates_df.copy()

# gets the length of the unprocessed dataframe
unprocessed_missing_ds = len(unprocessed_missing_df)

# find rows with missing instructions or outputs (empty inputs are permitted)
empty_rows_unprocessed = unprocessed_missing_df.index[
    unprocessed_missing_df['instruction'].isnull() | unprocessed_missing_df['instruction'].astype(str).str.strip().eq("") |
    unprocessed_missing_df['output'].isnull() | unprocessed_missing_df['output'].astype(str).str.strip().eq("")
].tolist()

# count empty rows per column within the unprocessed dataset
empty_instruction_rows_unprocessed = (unprocessed_missing_df['instruction'].isnull() | unprocessed_missing_df['instruction'].astype(str).str.strip().eq("")).sum()
empty_output_rows_unprocessed = (unprocessed_missing_df['output'].isnull() | unprocessed_missing_df['output'].astype(str).str.strip().eq("")).sum()

# gets the length of the missing rows
missing_row_count_unprocessed = len(empty_rows_unprocessed)

# drops the empty or missing rows from instruction and output columns only
processed_missing_df = unprocessed_missing_df.drop(index=empty_rows_unprocessed).reset_index(drop=True)

# gets the length of the processed dataframe
processed_missing_ds = len(processed_missing_df)

# count empty rows per column in the processed dataset
empty_instruction_rows_processed = (processed_missing_df['instruction'].isnull() | processed_missing_df['instruction'].astype(str).str.strip().eq("")).sum()
empty_output_rows_processed = (processed_missing_df['output'].isnull() | processed_missing_df['output'].astype(str).str.strip().eq("")).sum()

# prints a summary of the missing rows removal process
print(f"--- Missing Rows Summary ---")
print(f"    - Rows before : {unprocessed_missing_ds}")
print(f"    - Missing instruction (before) : {empty_instruction_rows_unprocessed}")
print(f"    - Missing output (before) : {empty_output_rows_unprocessed}")
print(f"    - Rows removed : {missing_row_count_unprocessed}")
print(f"    - Missing instruction (after) : {empty_instruction_rows_processed}")
print(f"    - Missing output (after) : {empty_output_rows_processed}")
print(f"    - Rows after : {processed_missing_ds}")

--- Missing Rows Summary ---
    - Rows before : 77007
    - Missing instruction (before) : 0
    - Missing output (before) : 0
    - Rows removed : 0
    - Missing instruction (after) : 0
    - Missing output (after) : 0
    - Rows after : 77007


**5. Fix Input PlaceHolders**

In [ ]:
# creates a copy of the dataframe
unprocessed_fix_placeholder_input_df = processed_missing_df.copy()

# list of placeholder strings to clean
placeholders = ['<noinput>', 'N/A', 'n/a', 'None', 'none']

# counts the number of rows with any of the placeholders before cleaning
pattern = '|'.join([re.escape(p) for p in placeholders])
placeholder_count_unprocessed = unprocessed_fix_placeholder_input_df['input'].str.contains(pattern, case=False, na=False).sum()

# replaces the placeholder strings with an actual empty string
for p in placeholders:
    unprocessed_fix_placeholder_input_df['input'] = unprocessed_fix_placeholder_input_df['input'].str.replace(p, '', case=False).str.strip()

# creates processed dataframe after cleaning
processed_fix_placeholder_input_df = unprocessed_fix_placeholder_input_df.copy()

# counts the number of rows with placeholders after cleaning
placeholder_count_processed = processed_fix_placeholder_input_df['input'].str.contains(pattern, case=False, na=False).sum()

# prints a summary of the cleaning process
print(f"--- Placeholders Summary ---")
print(f"    - Placeholders found (before) : {placeholder_count_unprocessed}")
print(f"    - Placeholders found (after)  : {placeholder_count_processed}")
print(f"    - Dataset rows : {len(processed_fix_placeholder_input_df)}")

--- Placeholders Summary ---
    - Placeholders found (before) : 18399
    - Placeholders found (after)  : 0
    - Dataset rows : 77007


**6. Remove Extra White Spaces**

In [ ]:
# creates another copy of the dataframe
unprocessed_spaces_df = processed_fix_placeholder_input_df.copy()

# REMOVE SPACES, NEW LINES, TABS AT START AND END OF ROWS

# count number of unprocessed rows with leading/trailing whitespace (spaces, tabs, newlines) per column before cleaning
leading_trailing_instruction_spaces_unprocessed = unprocessed_spaces_df['instruction'].str.contains(r'^\s+|\s+$', na=False).sum()
leading_trailing_input_spaces_unprocessed = unprocessed_spaces_df['input'].str.contains(r'^\s+|\s+$', na=False).sum()
leading_trailing_output_spaces_unprocessed = unprocessed_spaces_df['output'].str.contains(r'^\s+|\s+$', na=False).sum()

# removes extra spaces at the beginning and end (including newlines)
unprocessed_spaces_df = unprocessed_spaces_df.replace(r'^\s+|\s+$', '', regex=True)

# count number of processed rows with leading/trailing whitespace
leading_trailing_instruction_spaces_processed = unprocessed_spaces_df['instruction'].str.contains(r'^\s+|\s+$', na=False).sum()
leading_trailing_input_spaces_processed = unprocessed_spaces_df['input'].str.contains(r'^\s+|\s+$', na=False).sum()
leading_trailing_output_spaces_processed = unprocessed_spaces_df['output'].str.contains(r'^\s+|\s+$', na=False).sum()


# REMOVE TRAILING SPACES, TABS AT END OF LINES

# count number of unprocessed rows with internal trailing whitespace (spaces, tabs before a newline) per column before cleaning
internal_trailing_instruction_spaces_unprocessed = unprocessed_spaces_df['instruction'].str.contains(r'[ \t]+\n', na=False).sum()
internal_trailing_input_spaces_unprocessed = unprocessed_spaces_df['input'].str.contains(r'[ \t]+\n', na=False).sum()
internal_trailing_output_spaces_unprocessed = unprocessed_spaces_df['output'].str.contains(r'[ \t]+\n', na=False).sum()

# removes internal trailing whitespace (spaces, tabs before a newline) from all rows
unprocessed_spaces_df = unprocessed_spaces_df.replace(r'[ \t]+\n', '\n', regex=True)

# count number of processed rows with internal trailing whitespace (spaces, tabs before a newline) per column after cleaning
internal_trailing_instruction_spaces_processed = unprocessed_spaces_df['instruction'].str.contains(r'[ \t]+\n', na=False).sum()
internal_trailing_input_spaces_processed = unprocessed_spaces_df['input'].str.contains(r'[ \t]+\n', na=False).sum()
internal_trailing_output_spaces_processed = unprocessed_spaces_df['output'].str.contains(r'[ \t]+\n', na=False).sum()

# REMOVE EXTRA SPACES WITHIN CENTER OF TEXT

# count number of unprocessed rows with extra spaces and tabs in center per column before cleaning
extra_spaces_instruction_unprocessed = unprocessed_spaces_df['instruction'].str.contains(r'[ \t]{2,}', na=False).sum()
extra_spaces_input_unprocessed = unprocessed_spaces_df['input'].str.contains(r'[ \t]{2,}', na=False).sum()
extra_spaces_output_unprocessed = unprocessed_spaces_df['output'].str.contains(r'[ \t]{2,}', na=False).sum()

# removes extra spaces and tabs from all rows
processed_spaces_df = unprocessed_spaces_df.replace(r'[ \t]{2,}', ' ', regex=True)

# count number of processed rows with extra spaces and tabs in center per column after cleaning
extra_spaces_instruction_processed = processed_spaces_df['instruction'].str.contains(r'[ \t]{2,}', na=False).sum()
extra_spaces_input_processed = processed_spaces_df['input'].str.contains(r'[ \t]{2,}', na=False).sum()
extra_spaces_output_processed = processed_spaces_df['output'].str.contains(r'[ \t]{2,}', na=False).sum()

# gets the length of the processed dataframe
processed_spaces_ds = len(processed_spaces_df)

# prints a summary of the whitespace cleaning process
print(f"--- Whitespace Cleaning Summary ---")
print(f"\n")
print(f"    - Leading / trailing whitespace")
print(f"\n")
print(f"    - Rows with leading/trailing whitespace in instruction (before)  : {leading_trailing_instruction_spaces_unprocessed}")
print(f"    - Rows with leading/trailing whitespace in input (before)        : {leading_trailing_input_spaces_unprocessed}")
print(f"    - Rows with leading/trailing whitespace in output (before)       : {leading_trailing_output_spaces_unprocessed}")
print(f"    - Total rows unprocessed with leading/trailing whitespace        : {leading_trailing_instruction_spaces_unprocessed + leading_trailing_input_spaces_unprocessed + leading_trailing_output_spaces_unprocessed}")
print(f"    - Rows with leading/trailing whitespace in instruction (after)   : {leading_trailing_instruction_spaces_processed}")
print(f"    - Rows with leading/trailing whitespace in input (after)         : {leading_trailing_input_spaces_processed}")
print(f"    - Rows with leading/trailing whitespace in output (after)        : {leading_trailing_output_spaces_processed}")
print(f"    - Total rows processed with leading/trailing whitespace          : {leading_trailing_instruction_spaces_processed + leading_trailing_input_spaces_processed + leading_trailing_output_spaces_processed}")
print(f"\n")
print(f"    - Internal trailing whitespace")
print(f"\n")
print(f"    - Rows with internal trailing whitespace in instruction (before) : {internal_trailing_instruction_spaces_unprocessed}")
print(f"    - Rows with internal trailing whitespace in input (before)       : {internal_trailing_input_spaces_unprocessed}")
print(f"    - Rows with internal trailing whitespace in output (before)      : {internal_trailing_output_spaces_unprocessed}")
print(f"    - Total rows unprocessed with internal trailing whitespace       : {internal_trailing_instruction_spaces_unprocessed + internal_trailing_input_spaces_unprocessed + internal_trailing_output_spaces_unprocessed}")
print(f"    - Rows with internal trailing whitespace in instruction (after)  : {internal_trailing_instruction_spaces_processed}")
print(f"    - Rows with internal trailing whitespace in input (after)        : {internal_trailing_input_spaces_processed}")
print(f"    - Rows with internal trailing whitespace in output (after)       : {internal_trailing_output_spaces_processed}")
print(f"    - Total rows processed with internal trailing whitespace         : {internal_trailing_instruction_spaces_processed + internal_trailing_input_spaces_processed + internal_trailing_output_spaces_processed}")
print(f"\n")
print(f"    - Extra white spaces and tabs in center")
print(f"\n")
print(f"    - Extra spaces in instruction (before)                           : {extra_spaces_instruction_unprocessed}")
print(f"    - Extra spaces in input (before)                                 : {extra_spaces_input_unprocessed}")
print(f"    - Extra spaces in output (before)                                : {extra_spaces_output_unprocessed}")
print(f"    - Total unprocessed extra spaces and tabs                        : {extra_spaces_instruction_unprocessed + extra_spaces_input_unprocessed + extra_spaces_output_unprocessed}")
print(f"    - Extra spaces in instruction (after)                            : {extra_spaces_instruction_processed}")
print(f"    - Extra spaces in input (after)                                  : {extra_spaces_input_processed}")
print(f"    - Extra spaces in output (after)                                 : {extra_spaces_output_processed}")
print(f"    - Total processed extra spaces and tabs                          : {extra_spaces_instruction_processed + extra_spaces_input_processed + extra_spaces_output_processed}")
print(f"\n")
print(f"    - Total Dataset Rows                                             : {processed_spaces_ds}")

--- Whitespace Cleaning Summary ---


    - Leading / trailing whitespace


    - Rows with leading/trailing whitespace in instruction (before)  : 0
    - Rows with leading/trailing whitespace in input (before)        : 0
    - Rows with leading/trailing whitespace in output (before)       : 592
    - Total rows unprocessed with leading/trailing whitespace        : 592
    - Rows with leading/trailing whitespace in instruction (after)   : 0
    - Rows with leading/trailing whitespace in input (after)         : 0
    - Rows with leading/trailing whitespace in output (after)        : 0
    - Total rows processed with leading/trailing whitespace          : 0


    - Internal trailing whitespace


    - Rows with internal trailing whitespace in instruction (before) : 0
    - Rows with internal trailing whitespace in input (before)       : 139
    - Rows with internal trailing whitespace in output (before)      : 5176
    - Total rows unprocessed with internal trailing whitespace       : 53

**7. AlpaCare-52k Truncuted Rows (output)**

In [ ]:
# creates a copy of the dataframe
unprocessed_truncuted_df = processed_spaces_df.copy()

# gets the length of the unprocessed dataframe
unprocessed_truncuted_ds = len(unprocessed_truncuted_df)

# creates a mask for AlpaCare-52k rows that do not end with proper closing punctuation
drop_mask = (unprocessed_truncuted_df['dataset_source'] == 'AlpaCare-52k') & \
            (~unprocessed_truncuted_df['output'].str.strip().str.contains(r'[.!?)\]"]$', regex=True, na=False))

# gets the number of cut off rows before cleaning
unprocessed_truncuted_count = drop_mask.sum()

# drops the cut off rows by keeping only the ones that don't match the mask
processed_truncuted_df = unprocessed_truncuted_df[~drop_mask].reset_index(drop=True)

# gets the length of the processed dataframe
processed_truncuted_ds = len(processed_truncuted_df)

# verifies no cut off rows remain in AlpaCare after cleaning
processed_truncuted_count = (~processed_truncuted_df.loc[processed_truncuted_df['dataset_source'] == 'AlpaCare-52k', 'output'].str.strip().str.contains(r'[.!?)\]"]$', regex=True, na=False)).sum()

# prints a summary of the cut off detection process
print(f"--- Truncuted Rows Summary (AlpaCare-52k) ---")
print(f"    - Dataset Rows (before)        : {unprocessed_truncuted_ds}")
print(f"    - Cut off output rows (before) : {unprocessed_truncuted_count}")
print(f"    - Dataset Rows (after)         : {processed_truncuted_ds}")
print(f"    - Cut off output rows (after)  : {processed_truncuted_count}")

--- Truncuted Rows Summary (AlpaCare-52k) ---
    - Dataset Rows (before)        : 77007
    - Cut off output rows (before) : 17434
    - Dataset Rows (after)         : 59573
    - Cut off output rows (after)  : 0


**8. Remove AI Mentions**

In [ ]:
# creates a copy of the dataframe
unprocessed_ai_text_df = processed_truncuted_df.copy()

# list of ai patterns to remove
ai_patterns = [
    r'\bas an ai\b',
    r'\bas a language model\b'
]

# combines all columns into one string per row
combined = unprocessed_ai_text_df[['instruction', 'input', 'output']].astype(str).agg(' '.join, axis=1)

# count per pattern before removal
pattern_counts = {pattern: combined.str.contains(pattern, case=False, na=False).sum() for pattern in ai_patterns}

# drop rows that match any pattern and reset index
mask = combined.str.contains('|'.join(ai_patterns), case=False, na=False)
processed_ai_text_df = unprocessed_ai_text_df[~mask].reset_index(drop=True)

# get counts for specific phrases using regex
as_an_ai_count = pattern_counts[r'\bas an ai\b']
language_model_count = pattern_counts[r'\bas a language model\b']

# prints a summary of the ai words detection process
print(f"--- AI Words Detection Summary ---")
print(f"    - Dataset Rows (before)                       : {len(unprocessed_ai_text_df)}")
print(f"    - as an ai detected (before)                  : {as_an_ai_count}")
print(f"    - as a language model detected (before)       : {language_model_count}")
print(f"    - Rows removed in total                       : {len(unprocessed_ai_text_df) - len(processed_ai_text_df)}")
print(f"    - Dataset Rows (after)                        : {len(processed_ai_text_df)}")

--- AI Words Detection Summary ---
    - Dataset Rows (before)                       : 59573
    - as an ai detected (before)                  : 251
    - as a language model detected (before)       : 22
    - Rows removed in total                       : 273
    - Dataset Rows (after)                        : 59300


**9. Remove Rows, Over Token Limit**

In [ ]:
# load the Llama-3.2-3B-Instruct tokenizer to check token lengths
llama_tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-3B-Instruct")

# max allowed tokens
max_tokens = 2048

# creates a copy of the dataframe
unprocessed_llama_tokens_df = processed_ai_text_df.copy()

# gets the length of the unprocessed dataframe
unprocessed_llama_tokens_ds = len(unprocessed_llama_tokens_df)

# empty system prompt
system_prompt = ""

# function that get the total token count for a single row
def get_token_length(sample):
    # get instruction text
    user_prompt = str(sample["instruction"])
    # get input text if it exists
    input_text = sample.get("input", "")

    # add input to instruction if it's not empty
    if pd.notna(input_text) and str(input_text).strip():
        user_prompt = user_prompt + "\n\n" + str(input_text).strip()

    # structure the data as a list of messages
    messages = [
        {"role": "user", "content": system_prompt + user_prompt if system_prompt else user_prompt},
        {"role": "assistant", "content": str(sample['output'])}
    ]

    # use the auto template
    chat_string = llama_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    tokens = llama_tokenizer.encode(chat_string)

    return len(tokens)

# calculate token lengths for all rows
token_lengths = unprocessed_llama_tokens_df.apply(get_token_length, axis=1)

# keep only rows with equal to or less then 2048
processed_llama_tokens_df = unprocessed_llama_tokens_df[token_lengths <= max_tokens].reset_index(drop=True)

# gets the length of the processed dataframe
processed_llama_tokens_ds = len(processed_llama_tokens_df)

# calculate how many rows were dropped
dropped_rows = unprocessed_llama_tokens_ds - processed_llama_tokens_ds

# prints a summary of the token filter process
print(f"--- Llama-3.2-3B-Instruct Token Filter Summary ---")
print(f"    - Dataset Rows (before)                : {unprocessed_llama_tokens_ds}")
print(f"    - Longest row found (tokens)           : {token_lengths.max()}")
print(f"    - Rows above 2048 tokens removed        : {dropped_rows}")
print(f"    - Dataset Rows (after)                 : {processed_llama_tokens_ds}")

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

--- Llama-3.2-3B-Instruct Token Filter Summary ---
    - Dataset Rows (before)                : 59300
    - Longest row found (tokens)           : 1435
    - Rows above 2048 tokens removed        : 0
    - Dataset Rows (after)                 : 59300


In [ ]:
# load the Qwen2.5-3B-Instruct tokenizer to check token lengths
qwen_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-3B-Instruct")

# max allowed tokens
max_tokens = 2048

# creates a copy of the dataframe
unprocessed_qwen_tokens_df = processed_llama_tokens_df.copy()

# gets the length of the unprocessed dataframe
unprocessed_qwen_tokens_ds = len(unprocessed_qwen_tokens_df)

# empty system prompt
system_prompt = ""

# function that get the total token count for a single row
def get_qwen_token_length(sample):
    # get instruction text
    user_prompt = str(sample["instruction"])
    # get input text if it exists
    input_text = sample.get("input", "")

    # add input to instruction if it's not empty
    if pd.notna(input_text) and str(input_text).strip():
        user_prompt = user_prompt + "\n\n" + str(input_text).strip()

    # structure the data as a list of messages
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
        {"role": "assistant", "content": str(sample['output'])}
    ]

    # use the auto template
    chat_string = qwen_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    tokens = qwen_tokenizer.encode(chat_string)

    return len(tokens)

# calculate token lengths for all rows
qwen_token_lengths = unprocessed_qwen_tokens_df.apply(get_qwen_token_length, axis=1)

# keep only rows with equal to or less then 2048
processed_qwen_tokens_df = unprocessed_qwen_tokens_df[qwen_token_lengths <= max_tokens].reset_index(drop=True)

# gets the length of the processed dataframe
processed_qwen_tokens_ds = len(processed_qwen_tokens_df)

# calculate how many rows were dropped
dropped_rows_qwen = unprocessed_qwen_tokens_ds - processed_qwen_tokens_ds

# prints a summary of the token filter process
print(f"--- Qwen2.5-3B-Instruct Token Filter Summary ---")
print(f"    - Dataset Rows (before)                : {unprocessed_qwen_tokens_ds}")
print(f"    - Longest row found (tokens)           : {qwen_token_lengths.max()}")
print(f"    - Rows above 2048 tokens removed        : {dropped_rows_qwen}")
print(f"    - Dataset Rows (after)                 : {processed_qwen_tokens_ds}")

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

--- Qwen2.5-3B-Instruct Token Filter Summary ---
    - Dataset Rows (before)                : 59300
    - Longest row found (tokens)           : 1423
    - Rows above 2048 tokens removed        : 0
    - Dataset Rows (after)                 : 59300


**11. Download Random Sample**

In [ ]:
# saves 10 random rows from the final dataset to a text file to verify data quality
samples = processed_qwen_tokens_df.sample(10, random_state=42)

# opens a text file to save the samples
with open('medical_sample_verification.txt', 'w') as f:
    for count, (i, row) in enumerate(samples.iterrows(), start=1):
        f.write(f"\n--- Sample {count} ---\n")
        f.write(f"    - Instruction : {row['instruction']}\n")
        f.write(f"    - Input : {row['input']}\n")
        f.write(f"    - Output : {row['output']}\n")
        f.write(f"    - Subset : {row['dataset_source']}\n")
        f.write(f"\n{'='*80}\n")

# prints confirmation of the save
print("--- Saved to medical_sample_verification.txt ---")

--- Saved to medical_sample_verification.txt ---


**10. Calculate Prestnages of Subsets**

In [ ]:
# counts the number of rows for each dataset source
alpacare_count = (processed_qwen_tokens_df['dataset_source'] == 'AlpaCare-52k').sum()
biomed_count = (processed_qwen_tokens_df['dataset_source'] == 'Bioinstruct').sum()

# gets the total number of rows in the final dataset
final_ds_size = len(processed_qwen_tokens_df)

# calculates the percentage of each dataset source
alpacare_percentage = (alpacare_count / final_ds_size) * 100
biomed_percentage = (biomed_count / final_ds_size) * 100

# prints a summary of the dataset subset percentages
print(f"--- Dataset Percentages Summary ---")
print(f"    - AlpaCare-52k Size : {alpacare_count}")
print(f"    - Bioinstruct Size : {biomed_count}")
print(f"    - AlpaCare-52k Percentage : {round(alpacare_percentage, 2)}%")
print(f"    - Bioinstruct Percentage : {round(biomed_percentage, 2)}%")
print(f"    - Final Size : {final_ds_size}")

--- Dataset Percentages Summary ---
    - AlpaCare-52k Size : 34295
    - Bioinstruct Size : 25005
    - AlpaCare-52k Percentage : 57.83%
    - Bioinstruct Percentage : 42.17%
    - Final Size : 59300


**12. Split Dataset**

In [ ]:
# split the dataset into train 90% and val 5% and test 5%
medical_train, temp = train_test_split(processed_qwen_tokens_df, test_size=0.1, random_state=42)
medical_val, medical_test   = train_test_split(temp, test_size=0.5, random_state=42)

# gets the total number of rows in the final dataset
final_ds = len(processed_qwen_tokens_df)

# calculate percentages
train_percentage = (len(medical_train) / final_ds) * 100
val_percentage = (len(medical_val) / final_ds) * 100
test_percentage = (len(medical_test) / final_ds) * 100

# prints a summary of the dataset set percentages and size
print(f"--- Mathematical Split Summary ---")
print(f"    - Final Dataset Size Combined : {final_ds}")
print(f"    - Columns : {processed_qwen_tokens_df.columns.tolist()}")
print(f"    - Train Set Size : {len(medical_train)}")
print(f"    - Train Set Percentage : {round(train_percentage, 2)}%")
print(f"    - Val Set Size : {len(medical_val)}")
print(f"    - Val Set Percentage : {round(val_percentage, 2)}%")
print(f"    - Test Set Size : {len(medical_test)}")
print(f"    - Test Set Percentage : {round(test_percentage, 2)}%")

--- Mathematical Split Summary ---
    - Final Dataset Size Combined : 59300
    - Columns : ['instruction', 'input', 'output', 'dataset_source']
    - Train Set Size : 53370
    - Train Set Percentage : 90.0%
    - Val Set Size : 2965
    - Val Set Percentage : 5.0%
    - Test Set Size : 2965
    - Test Set Percentage : 5.0%


**13. Export the Final Dataset (Mathematical Version)**

In [ ]:
# folder path to save to in goolge drive
folder_path = "/content/drive/MyDrive/Fine-Tuning-Datasets/Med-MathInstruct/Medical-Data-Only"

# creates folder if does not exist
os.makedirs(folder_path, exist_ok=True)

# deletes existing files before saving to ensure new one is saved
for f in ["mv_train.jsonl", "mv_val.jsonl", "mv_test.jsonl"]:
    file_path = f"{folder_path}/{f}"
    if os.path.exists(file_path):
        os.remove(file_path)

# exports the data to the file location on google drive
medical_train.to_json(f"{folder_path}/mv_train.jsonl", orient="records", lines=True)
medical_val.to_json(f"{folder_path}/mv_val.jsonl", orient="records", lines=True)
medical_test.to_json(f"{folder_path}/mv_test.jsonl", orient="records", lines=True)

# print confirmation that the files have been saved
print(f"--- Mathematical Version Files Saved ---")
print(f"    - Columns: {medical_train.columns.tolist()}")
print(f"    - Saved mathematical version training dataset successfully")
print(f"    - Saved mathematical version val dataset successfully")
print(f"    - Saved mathematical version test dataset successfully")

--- Mathematical Version Files Saved ---
    - Columns: ['instruction', 'input', 'output', 'dataset_source']
    - Saved mathematical version training dataset successfully
    - Saved mathematical version val dataset successfully
    - Saved mathematical version test dataset successfully


**14. Remove subset_column**

In [ ]:
# removes the column called 'dataset_source' from train set
medical_train = medical_train.drop('dataset_source', axis=1, errors='ignore')
medical_val = medical_val.drop('dataset_source', axis=1, errors='ignore')
medical_test = medical_test.drop('dataset_source', axis=1, errors='ignore')

# prints confirmation
print(f"--- Dataset Remove Column dataset_source ---")
print(f"    - Train Columns: {medical_train.columns.tolist()}")
print(f"    - Val Columns:   {medical_val.columns.tolist()}")
print(f"    - Test Columns:  {medical_test.columns.tolist()}")

--- Dataset Remove Column dataset_source ---
    - Train Columns: ['instruction', 'input', 'output']
    - Val Columns:   ['instruction', 'input', 'output']
    - Test Columns:  ['instruction', 'input', 'output']


**15. Export the Final Dataset (Non Mathematical)**

In [ ]:
# folder path to save to in goolge drive
folder_path = "/content/drive/MyDrive/Fine-Tuning-Datasets/MedInstruct"

# creates folder if does not exist
os.makedirs(folder_path, exist_ok=True)

# deletes existing files before saving
for f in ["medical_train.jsonl", "medical_val.jsonl", "medical_test.jsonl"]:
    file_path = f"{folder_path}/{f}"
    if os.path.exists(file_path):
        os.remove(file_path)

# exports the data to the file location on google drive
medical_train.to_json(f"{folder_path}/medical_train.jsonl", orient="records", lines=True)
medical_val.to_json(f"{folder_path}/medical_val.jsonl", orient="records", lines=True)
medical_test.to_json(f"{folder_path}/medical_test.jsonl", orient="records", lines=True)

# print confirmation that the files have been saved
print(f"--- Non Mathematical Dataset Files Saved ---")
print(f"    - Columns: {medical_train.columns.tolist()}")
print(f"    - Saved non-mathematical training dataset successfully")
print(f"    - Saved non-mathematical val dataset successfully")
print(f"    - Saved non-mathematical test dataset successfully")

--- Non Mathematical Dataset Files Saved ---
    - Columns: ['instruction', 'input', 'output']
    - Saved non-mathematical training dataset successfully
    - Saved non-mathematical val dataset successfully
    - Saved non-mathematical test dataset successfully
